# Exploration — prime pure incendie

Lecture du train local (`data/raw/`, non versionné). Les calculs sont
dans `agri_fire.eda` et les graphes dans `agri_fire.plots`. Ce notebook
les appelle.

Lancer depuis la racine du dépôt. Les 697 Mo ne sont pas relus par les tests.

In [1]:
from pathlib import Path

import pandas as pd

from agri_fire.eda import (
    claim_absence,
    constant_columns,
    duplicate_pairs,
    eda_columns,
    exposure_gap,
    exposure_profile,
    matching_nonzero_support,
    summarize_exercise_years,
    summarize_zones,
    weather_rank_agreement,
)
from agri_fire.plots import (
    save_claim_share_by_year,
    save_exposure_histogram,
    save_weather_rank_strip,
    save_year_volume,
    save_zone_rates,
)

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent

train_path = root / "data" / "raw" / "train_input_Z61KlZo.csv"
target_path = root / "data" / "raw" / "train_output_DzPxaPY.csv"
picture_dir = root / "pictures" / "presentations"

header = pd.read_csv(train_path, nrows=0).columns.tolist()
columns = eda_columns(header)
features = pd.read_csv(train_path, usecols=columns, low_memory=False)
targets = pd.read_csv(target_path)
print(f"{len(features)} lignes, {len(columns)} colonnes lues")

383610 lignes, 198 colonnes lues


## Sans sinistre, et exposition

`FREQ` = 0 veut dire aucun sinistre incendie sur la ligne. L'exposition
est `ANNEE_ASSURANCE` : la même colonne dans les variables et dans les cibles.

In [2]:
absence = claim_absence(targets)
profile = exposure_profile(targets)
gap = exposure_gap(features, targets)
print(absence)
print(profile)
print("écart d'exposition entre les deux fichiers", gap)

ClaimAbsence(n_rows=383610, n_without_claim=380716, share_without_claim=0.9924558796694559, n_with_claim=2894, n_negative_frequency=0, n_zero_charge=381252, n_negative_charge=6, n_zero_cost_with_claim=536)
ExposureProfile(n_rows=383610, total_exposure=269124.2080169793, total_claim_count=3004.0, claims_per_exposure_year=0.011162132244195867, share_partial=0.547193764500404, share_full_year=0.44566617137196635, share_longer=0.007140064127629624, min_exposure=0.0027322404, median_exposure=0.8821917808, mean_exposure=0.7015568103463916, max_exposure=2.0)
écart d'exposition entre les deux fichiers 0.0


## Millésimes et zones

`AN_EXERC` vaut `ANNEE1` à `ANNEE9`. L'ordre du tableau suit le numéro
du libellé, pas une date. `ZONE` est le lieu du risque.

In [3]:
by_year = summarize_exercise_years(features, targets)
by_zone = summarize_zones(features, targets)
print(by_year.to_string(index=False))
print("zones", by_zone["zone"].nunique())
print(
    "effectif min / médiane / max",
    int(by_zone["n_contracts"].min()),
    float(by_zone["n_contracts"].median()),
    int(by_zone["n_contracts"].max()),
)
big = by_zone[by_zone["n_contracts"] >= 1000]
print(
    "zones d'au moins 1000 contrats",
    len(big),
    "taux min / max",
    float(big["claim_rate"].min()),
    float(big["claim_rate"].max()),
)
print("zones de moins de 100 contrats", int((by_zone["n_contracts"] < 100).sum()))

exercise  n_contracts     exposure  n_without_claim  n_with_claim  mean_charge  total_charge  share_without_claim  share_with_claim
  ANNEE1        67334 47502.301370            66778           556   191.990207   12927468.60             0.991743          0.008257
  ANNEE2        31535 21611.931694            31339           196   129.649508    4088497.24             0.993785          0.006215
  ANNEE3        30805 21815.871233            30479           326   256.789484    7910400.06             0.989417          0.010583
  ANNEE4        30591 21487.065753            30299           292   161.541650    4941720.62             0.990455          0.009545
  ANNEE5        26573 19211.800000            26287           286   146.709375    3898508.22             0.989237          0.010763
  ANNEE6        77819 54463.816438            77338           481   181.499663   14124122.27             0.993819          0.006181
  ANNEE7        42378 29397.167123            42042           336   211.6618

## Blocs redondants

Capitaux, surfaces, et météo en moyenne / maximum / somme. Une paire
n'est un doublon que si les valeurs actives coïncident, pas seulement
les zéros. Le rang météo est le préfixe `01` à `04` du libellé.

In [4]:
surfaces = [name for name in features.columns if name.startswith("SURFACE")]
capitals = [name for name in features.columns if name.startswith("KAPITAL")]

surface_copies = duplicate_pairs(features, surfaces)
capital_copies = duplicate_pairs(features, capitals)
surface_constants = constant_columns(features, surfaces)
capital_support = matching_nonzero_support(features, capitals)
weather = weather_rank_agreement(features)

print("surfaces en double")
print(surface_copies.to_string(index=False))
print("surfaces constantes")
print(surface_constants.to_string(index=False))
print("capitaux en double")
print(capital_copies.to_string(index=False))
print("capitaux au même support non nul")
print(capital_support.to_string(index=False))

with_sum = weather.dropna(subset=["same_rank_mean_sum"])
print(
    "grandeurs avec une somme",
    len(with_sum),
    "rang identique à la moyenne",
    int(with_sum["same_rank_mean_sum"].eq(1).sum()),
)
print("corrélation max–moyenne, médiane", float(weather["corr_mean_max"].median()))

surfaces en double
    left    right  n_both  n_differ  share_equal  n_active  share_equal_active
SURFACE1 SURFACE2  383610         5     0.999987    356223            0.999986
SURFACE4 SURFACE6  375382         0     1.000000    300008            1.000000
surfaces constantes
Empty DataFrame
Columns: [column, value, n_observed, n_missing]
Index: []
capitaux en double
     left     right  n_both  n_differ  share_equal  n_active  share_equal_active
KAPITAL10 KAPITAL11  264605         0          1.0    153254                 1.0
capitaux au même support non nul
     left     right  n_nonzero
 KAPITAL1 KAPITAL13       9246
KAPITAL10 KAPITAL11     153254
grandeurs avec une somme 30 rang identique à la moyenne 30
corrélation max–moyenne, médiane 0.8425811633424805


## Graphes

Fond crème, le même que les diapositives. Un fichier par langue quand
le libellé d'axe est un mot.

In [5]:
save_claim_share_by_year(by_year, picture_dir / "part-sinistre-annee-fr.png", language="fr")
save_claim_share_by_year(by_year, picture_dir / "part-sinistre-annee-en.png", language="en")
save_year_volume(by_year, picture_dir / "volume-annees-fr.png", language="fr")
save_year_volume(by_year, picture_dir / "volume-annees-en.png", language="en")
save_zone_rates(by_zone, picture_dir / "zones-fr.png", language="fr")
save_zone_rates(by_zone, picture_dir / "zones-en.png", language="en")
save_exposure_histogram(
    targets["ANNEE_ASSURANCE"],
    picture_dir / "exposition-fr.png",
    language="fr",
)
save_exposure_histogram(
    targets["ANNEE_ASSURANCE"],
    picture_dir / "exposition-en.png",
    language="en",
)
save_weather_rank_strip(weather, picture_dir / "meteo-rangs-fr.png", language="fr")
save_weather_rank_strip(weather, picture_dir / "meteo-rangs-en.png", language="en")
print("graphes écrits dans", picture_dir)

graphes écrits dans C:\Users\DimiP\Documents\GitHub\4_Projects\ML-zoom-camp-midterm-project\pictures\presentations


## Lecture

Sur les 383 610 contrats, 380 716 n'ont aucun sinistre (99,25 %).
2 894 en ont au moins un. Parmi eux, 536 ont un coût moyen nul et 6
un coût négatif. Prédire une charge nulle partout donne un RMSE
d'environ 6 803 : le plancher vient de cette masse de zéros.

L'exposition médiane est 0,88 an. 54,7 % des contrats couvrent moins
d'un an, 44,6 % exactement un an, 0,7 % davantage (jusqu'à 2). Les deux
fichiers portent la même exposition. On compte 3 004 sinistres pour
269 124 années couvertes, soit environ 1,1 sinistre pour 100 années.

Les neuf millésimes ont tous entre 98,9 % et 99,5 % de contrats sans
sinistre. Les effectifs, eux, vont de 26 573 (`ANNEE5`) à 77 819
(`ANNEE6`).

Il y a 93 zones. Trois en ont moins de 100 contrats (la plus petite : 4).
Parmi les 86 zones d'au moins 1 000 contrats, le taux de sinistre va
d'environ 0,3 % à 1,9 %, soit un facteur six.

`SURFACE1` et `SURFACE2` diffèrent sur 5 contrats. `SURFACE4` et
`SURFACE6` sont identiques sur les lignes où les deux sont remplies.
`SURFACE14`, `SURFACE15` et `SURFACE20` ne sortent de zéro que sur 30, 31 et 15 contrats.
`KAPITAL10` et `KAPITAL11` sont égaux là où `KAPITAL11` est rempli (264 605 lignes) ; ailleurs `KAPITAL10` vaut 0.
`KAPITAL1` (0/1) est positif exactement quand `KAPITAL13` l'est (5 000 ou 10 000).

Trente grandeurs météo ont une somme. Son rang `01`–`04` est celui de
la moyenne sur chaque ligne comparée. Le maximum n'est pas une copie :
la corrélation médiane des rangs avec la moyenne est d'environ 0,84.